# Test Prediction

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = 'https://github.com/Tienkhoaa2908/ISAS2026-DLCS.git'
REPO_NAME = 'ISAS2026-DLCS'

if Path('/content').exists():
    work = Path('/content') / REPO_NAME
    if not work.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(work)], check=True)
else:
    cwd = Path.cwd()
    work = cwd if (cwd / 'requirements.txt').exists() else cwd.parent

work = work.resolve()
private_root = Path(os.environ['ISAS2026_PRIVATE_DIR']).expanduser().resolve() if os.environ.get('ISAS2026_PRIVATE_DIR') else (
    Path('/content/ISAS2026-private') if Path('/content').exists() else work.parent / f'{work.name}-private'
)
private_root = private_root.resolve()
if private_root == work or work in private_root.parents:
    raise ValueError('ISAS2026_PRIVATE_DIR must be outside the repository worktree.')
private_root.mkdir(parents=True, exist_ok=True)
cache_root = private_root / 'cache'
for name in ('pip', 'matplotlib', 'joblib'):
    (cache_root / name).mkdir(parents=True, exist_ok=True)
os.environ['PIP_CACHE_DIR'] = str(cache_root / 'pip')
os.environ['MPLCONFIGDIR'] = str(cache_root / 'matplotlib')
os.environ['JOBLIB_TEMP_FOLDER'] = str(cache_root / 'joblib')
os.chdir(work)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repository =', Path.cwd())


## Data preparation

In [ ]:
from pathlib import Path
import os, subprocess, sys

archive = Path(os.environ.get('DATA_ARCHIVE', str(private_root / 'data.zip'))).expanduser().resolve()
manifest = Path(os.environ.get('DATA_MANIFEST', str(private_root / 'data_manifest.json'))).expanduser().resolve()
for name, path in (('DATA_ARCHIVE', archive), ('DATA_MANIFEST', manifest)):
    if path == work or work in path.parents:
        raise ValueError(f'{name} must point outside the repository worktree.')

if not archive.is_file() or not manifest.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Set DATA_ARCHIVE and DATA_MANIFEST to input files outside the repository worktree.') from exc
    upload_cwd = Path.cwd()
    try:
        os.chdir(private_root)
        uploaded = files.upload()
    finally:
        os.chdir(upload_cwd)
    try:
        if not archive.is_file():
            archive = private_root / Path(next(name for name in uploaded if Path(name).suffix.lower() == '.zip')).name
        if not manifest.is_file():
            manifest = private_root / Path(next(name for name in uploaded if Path(name).suffix.lower() == '.json')).name
    except StopIteration as exc:
        raise FileNotFoundError('Upload both the input ZIP archive and manifest, or set DATA_ARCHIVE and DATA_MANIFEST.') from exc

run_root = private_root / 'runtime'
runtime = run_root / 'Dataset'
run_root.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'scripts/prepare_data.py',
    '--archive', str(archive), '--manifest', str(manifest), '--out', str(runtime)
], check=True)
print('data_root =', runtime)


## Validation reference

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

agg = pd.read_csv('results/lodo/aggregate.csv')
by_day = pd.read_csv('results/lodo/by_day.csv')
cm = pd.read_csv('results/lodo/confusion_main_centered_normalized.csv', index_col=0)
per_class = pd.read_csv('results/lodo/per_class_main_centered.csv')

display(agg)
display(by_day)
display(per_class)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm.to_numpy(), vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(cm.columns)), cm.columns, rotation=90)
ax.set_yticks(range(len(cm.index)), cm.index)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('Fixed-22-class LODO — main + centered smoothing')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## Unlabeled-input inference

In [ ]:
import subprocess, sys, pandas as pd, hashlib
from pathlib import Path

out = run_root / 'test_inference' / 'submission.csv'
subprocess.run([
    sys.executable, 'scripts/test_inference.py',
    '--data-root', str(runtime),
    '--out', str(out),
    '--workdir', str(run_root / 'test_inference')
], check=True)

submission = pd.read_csv(out)
sha256 = hashlib.sha256(out.read_bytes()).hexdigest()
expected = '4b1a80a214b53b531a40467a515e2afbe4350b7b0533144ec1415c3eb9a43d80'

print('rows =', len(submission))
print('missing_predictions =', int(submission['Location'].isna().sum()))
print('sha256 =', sha256)
print('matches_locked_submission =', sha256 == expected)
